# Optimization Mechanics (Learning Rates, Gradient Accumulation, and Loss Dynamics)

Hyperparameter selection and execution scheduling in Supervised Fine-Tuning (SFT) directly dictate whether optimization converges smoothly or causes gradient exploding, catastrophic forgetting, or VRAM starvation.

1. Learning Rate Dynamics & Warmup SchedulesUnlike pre-training from scratch where initial learning rates are high ($\sim 10^{-3}$ to $10^{-4}$), fine-tuning starts from an already optimized parameter distribution $\theta_0$. Large gradient steps will destroy pre-trained features (catastrophic forgetting).

Cosine Decay with Linear WarmupThe standard learning rate schedule for SFT is a linear warmup over $W$ steps followed by a cosine decay down to a minimum learning rate $\eta_{\min} = 0.1 \cdot \eta_{\max}$.$$\eta(t) = \begin{cases} \eta_{\max} \cdot \frac{t}{W} & \text{if } t \le W \\ \eta_{\min} + \frac{1}{2}(\eta_{\max} - \eta_{\min})\left(1 + \cos\left(\pi \frac{t - W}{T - W}\right)\right) & \text{if } t > W \end{cases}$$Why Warmup is Non-Negotiable: At step $0$, the task-specific classification head or LoRA adapters are initialized randomly or with zero weights. Initial gradients $\nabla_\theta \mathcal{L}$ are high-variance. Warmup prevents early gradient updates from destabilizing the base model's attention manifolds.Typical Learning Rate Scale for SFT:Full Fine-Tuning: $1 \times 10^{-6}$ to $2 \times 10^{-5}$PEFT / LoRA (Trainable parameters $\ll$ Total parameters): $1 \times 10^{-4}$ to $5 \times 10^{-4}$ (Higher rates are required to train low-rank matrices efficiently).